# S13-T1: YOLOv8n-seg fine-tune, measured gate, OpenVINO FP16 export

Fully automated, reproducible off-box run for **AC10-AC16** of `docs/sprints/s12_t5_spec.md`.
It fine-tunes on a CUDA GPU for 100 epochs with seed 42, validates `best.pt` and `last.pt`
on the frozen val split, writes a measured `val_metrics.json`, evaluates the retraining
gate, exports a real OpenVINO FP16 IR with `head_status: trained` / `reinitialised_tensors: 0`,
and builds a deterministic deployment zip.

**This notebook fails closed.** It never fabricates a dataset, never accepts a CPU/MPS run,
never copies the canonical v2 constants, and never writes a bundle when the gate fails.

| Requirement | Value |
| --- | --- |
| Runtime | Colab GPU: T4, L4 or better (>= 8 GB VRAM) |
| Disk | >= 30 GB free |
| Dataset | Materialized Dataset v3 (`dataset.yaml`, `split.json`, `manifest.json`, `images/`, `labels/`) |
| Wall clock | Roughly 2-4 h of GPU time for 100 epochs, plus validation and export |

## Inputs

Set these before running, or edit the config cell:

| Variable | Meaning |
| --- | --- |
| `KV_DATASET_DIR` | Directory holding a materialized Dataset v3 (default `/content/datasets/yolo_seg`) |
| `KV_DATASET_ZIP` | Optional zip of that directory, e.g. on mounted Drive |
| `KV_DATASET_DATES` | Optional comma-separated dates; materializes from R2 when `KV_DATASET_DIR` is absent |
| `KV_LABELS_DIR` | Optional human-reviewed label directory for R2 materialization (default `data/seg_prelabels`) |
| `R2_ENDPOINT`, `R2_BUCKET_NAME`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY` | Required only for R2 materialization; never paste them into this notebook |
| `KV_LATENCY_EVIDENCE` | Optional `bench_seg_latency.py` JSON from the AVX2 worker; binds the gate p95 to this IR |
| `KV_DRIVE_DIR` | Optional Drive directory to copy the finished bundle into |

The dataset is the only non-reproducible input: it is the human-reviewed, frozen Dataset v3
produced by `tools/eval/materialize_seg_dataset.py`. The notebook validates its split and
manifest digests before spending GPU time.

## 1. Install dependencies

Colab already ships CUDA-enabled PyTorch. Pinning `ultralytics` below 9 keeps the exporter's
`quantize=16` / `dynamic=True` contract intact.

In [ ]:
%pip install -q "ultralytics>=8.3,<9" "openvino>=2024.0" pillow pyyaml

import importlib
import platform
import sys

import torch

for name in ("ultralytics", "openvino", "PIL", "yaml", "torch"):
    module = importlib.import_module(name)
    print(f"{name:12s} {getattr(module, '__version__', 'unknown')}")
print("python      ", sys.version.split()[0], platform.platform())
print("cuda        ", torch.cuda.is_available(), torch.version.cuda)

## 2. Clone the repository and pin the training branch

The run is evidence only against a specific commit. The driver refuses a dirty working
tree; this cell prints `git status` so an accidental local edit is visible before training.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get("KV_REPO_URL", "https://github.com/529-studio/projectKV.git")
BRANCH = os.environ.get("KV_BRANCH", "feat/s13-t1-gpu-finetune-and-avx2-soak")
REPO = Path("/content/projectKV")

def run(cmd, cwd=None):
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], cwd=cwd, check=True)

try:
    if (REPO / ".git").is_dir():
        run(["git", "fetch", "origin", BRANCH], cwd=REPO)
        run(["git", "checkout", BRANCH], cwd=REPO)
        run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO)
    else:
        run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO])
except subprocess.CalledProcessError as exc:
    raise SystemExit(
        f"Cannot check out {BRANCH} from {REPO_URL}: {exc}. Push the branch first, "
        f"or set KV_BRANCH to a branch that exists on origin."
    ) from exc

os.chdir(REPO)
run(["git", "rev-parse", "HEAD"])
run(["git", "status", "--porcelain"])
print("working tree is", "dirty" if subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout.strip() else "clean")

## 3. Configure the run

`KV_EPOCHS` defaults to 100 and `KV_SEED` to 42. The gate requires at least 50 completed
epochs; a shorter run is a diagnostic, not evidence.

In [ ]:
import os
from pathlib import Path

KV_DATASET_DIR = Path(os.environ.get("KV_DATASET_DIR", "/content/datasets/yolo_seg"))
KV_DATASET_ZIP = Path(os.environ["KV_DATASET_ZIP"]) if os.environ.get("KV_DATASET_ZIP") else None
KV_DATASET_DATES = os.environ.get("KV_DATASET_DATES", "").strip()
KV_LABELS_DIR = os.environ.get("KV_LABELS_DIR", "").strip()
KV_WORK_DIR = Path(os.environ.get("KV_WORK_DIR", "/content/kv_s13"))
KV_EXPORT_DIR = os.environ.get("KV_EXPORT_DIR", "models/yolov8n_seg_openvino_416")
KV_LATENCY_EVIDENCE = os.environ.get("KV_LATENCY_EVIDENCE", "").strip()
KV_EPOCHS = int(os.environ.get("KV_EPOCHS", "100"))
KV_SEED = int(os.environ.get("KV_SEED", "42"))
KV_OVERWRITE = os.environ.get("KV_OVERWRITE", "1") == "1"

assert KV_EPOCHS >= 50, "AC10 requires at least 50 completed epochs"
assert KV_SEED == 42, "the frozen split was created with seed 42"
print({
    "dataset": str(KV_DATASET_DIR),
    "work_dir": str(KV_WORK_DIR),
    "export_dir": KV_EXPORT_DIR,
    "epochs": KV_EPOCHS,
    "seed": KV_SEED,
    "latency_evidence": KV_LATENCY_EVIDENCE or "<provisional local p95>",
})

## 4. GPU and disk pre-flight

The driver repeats these checks, but failing here avoids wasting a dataset download on a
CPU runtime. MPS and CPU runs are explicitly not gate-eligible (spec §2.6).

In [ ]:
import platform
import shutil

import torch

assert platform.system() == "Linux", f"off-box evidence requires Linux, got {platform.system()}"
assert platform.machine().lower() in {"x86_64", "amd64"}, platform.machine()
assert torch.cuda.is_available(), (
    "No CUDA GPU. In Colab choose Runtime > Change runtime type > T4/L4/A100. "
    "CPU and MPS training are not accepted as evidence."
)

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
assert vram_gb >= 8.0, f"{props.name} has {vram_gb:.1f} GB VRAM; need >= 8 GB"
assert free_gb >= 30.0, f"only {free_gb:.1f} GB free; need >= 30 GB"

!nvidia-smi
print(f"{props.name}: {vram_gb:.1f} GB VRAM, {free_gb:.1f} GB free disk")

## 5. Acquire and verify Dataset v3

Accepted sources, in order:

1. an existing `KV_DATASET_DIR`;
2. a `KV_DATASET_ZIP` (mounted Drive or uploaded file);
3. `KV_DATASET_DATES` plus R2 credentials, materialized by `tools/eval/materialize_seg_dataset.py`.

The dataset must carry `split.json` and `manifest.json`: without them the weights cannot be
bound to a frozen split, and AC10/AC14 cannot be met.

In [ ]:
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

def flatten_single_root(root: Path) -> None:
    """A Drive zip often wraps the dataset in one folder; lift it into place."""
    if (root / "dataset.yaml").is_file():
        return
    children = [
        child
        for child in root.iterdir()
        if child.is_dir() and child.name != "__MACOSX"
    ]
    if len(children) == 1 and (children[0] / "dataset.yaml").is_file():
        inner = children[0]
        for item in list(inner.iterdir()):
            shutil.move(str(item), str(root / item.name))
        inner.rmdir()

if not (KV_DATASET_DIR / "dataset.yaml").is_file():
    if KV_DATASET_ZIP is not None:
        assert KV_DATASET_ZIP.is_file(), f"dataset zip not found: {KV_DATASET_ZIP}"
        shutil.rmtree(KV_DATASET_DIR, ignore_errors=True)
        KV_DATASET_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(KV_DATASET_ZIP) as archive:
            archive.extractall(KV_DATASET_DIR)
        flatten_single_root(KV_DATASET_DIR)
    elif KV_DATASET_DATES:
        command = [
            sys.executable,
            "tools/eval/materialize_seg_dataset.py",
            "--dates", KV_DATASET_DATES,
            "--out", str(KV_DATASET_DIR),
            "--val-split", "0.2",
            "--seed", "42",
            "--manifest", str(KV_WORK_DIR / "dataset_manifest.json"),
            "--replace",
        ]
        if KV_LABELS_DIR:
            command += ["--labels-dir", KV_LABELS_DIR]
        print("$", " ".join(command))
        subprocess.run(command, check=True)
    else:
        raise SystemExit(
            "No dataset. Set KV_DATASET_DIR, KV_DATASET_ZIP or "
            "KV_DATASET_DATES + R2_* credentials. Synthetic data is forbidden."
        )

required = ["dataset.yaml", "split.json", "manifest.json"]
missing = [name for name in required if not (KV_DATASET_DIR / name).is_file()]
assert not missing, f"dataset is missing {missing}; it is not a materialized Dataset v3"
for split in ("train", "val"):
    images = list((KV_DATASET_DIR / "images" / split).glob("*.jpg"))
    labels = list((KV_DATASET_DIR / "labels" / split).glob("*.txt"))
    print(f"{split:5s} images={len(images)} labels={len(labels)}")
    assert images and len(images) == len(labels), f"{split} split is empty or has orphans"

## 6. Fine-tune, validate, gate and export

This invokes `tools/train/colab_pipeline.py`, which:

- runs `train_yolov8_seg.py` with `--epochs 100 --seed 42 --device 0` and the spec's
  fixed hyperparameters (AdamW, 416 px, batch 16, patience 25, AMP, deterministic);
- validates `best.pt` and `last.pt` with `model.val(split="val", conf=0.25, iou=0.5)`;
- accumulates per-class **mask-pixel IoU** from the validator's mask batches;
- writes `val_metrics.json`, validates it with `load_val_metrics`, and binds it to
  `run_manifest.json`;
- measures the provisional IR with `verify_openvino_ir` and evaluates
  `evaluate_retraining_gate()`;
- re-exports with `--metrics` provenance and asserts `head_status: trained`,
  `reinitialised_tensors: 0`, plus matching training SHAs;
- writes the deterministic bundle.

If `KV_LATENCY_EVIDENCE` points at an AVX2 worker benchmark for this IR, that p95 is used.
Otherwise the gate gets a clearly labelled provisional CPU number and the post-run cell
explains the remaining promotion step.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

command = [
    sys.executable, "tools/train/colab_pipeline.py",
    "--data-dir", str(KV_DATASET_DIR),
    "--work-dir", str(KV_WORK_DIR),
    "--export-dir", KV_EXPORT_DIR,
    "--epochs", str(KV_EPOCHS),
    "--seed", str(KV_SEED),
    "--gpu-index", "0",
    "--batch", "16",
    "--imgsz", "416",
    "--workers", "4",
]
if KV_LATENCY_EVIDENCE:
    command += ["--latency-evidence", KV_LATENCY_EVIDENCE]
if KV_OVERWRITE:
    command += ["--overwrite"]

print("$", " ".join(command))
subprocess.run(command, cwd=Path.cwd(), env=os.environ.copy(), check=True)

## 7. Inspect the measured evidence

This cell re-reads the artifacts the driver produced and asserts the AC14 invariants a
second time, independently of the export call. It also prints `last.pt` beside `best.pt`
(AC13) so the best-epoch margin is visible, not just claimed.

In [ ]:
import json
from pathlib import Path

import yaml

RUN_DIR = KV_WORK_DIR / "runs" / "segment" / "train_yolov8n_seg"
EXPORT = Path.cwd() / KV_EXPORT_DIR
VAL_METRICS = json.loads((RUN_DIR / "val_metrics.json").read_text(encoding="utf-8"))
GATE = json.loads((RUN_DIR / "gate_summary.json").read_text(encoding="utf-8"))
METADATA = yaml.safe_load((EXPORT / "metadata.yaml").read_text(encoding="utf-8"))
PROJECTKV = METADATA["projectkv"]

assert GATE["verdict"] == "PASSED", GATE
assert PROJECTKV["head_status"] == "trained", PROJECTKV.get("head_status")
assert PROJECTKV["reinitialised_tensors"] == 0, PROJECTKV.get("reinitialised_tensors")
assert PROJECTKV["class_count"] == 6 and PROJECTKV["output_channels"] == 42
assert "training" in PROJECTKV, "metadata has no training provenance block"
assert PROJECTKV["training"]["best_pt_sha256"] == VAL_METRICS["weights_sha256"]
assert PROJECTKV["training"]["epochs"] == VAL_METRICS["epochs_completed"]

print("gate              ", GATE["verdict"])
print("metrics provenance", GATE["metrics_provenance"])
print("latency provenance", GATE["latency_provenance"])
print("epochs            ", VAL_METRICS["epochs_completed"], "best", VAL_METRICS["best_epoch"])
print("stability margin  ", VAL_METRICS["stability"]["best_minus_median"])
print()
print("class              best IoU   last.pt IoU")
for name in VAL_METRICS["class_iou"]:
    last = (VAL_METRICS.get("last_pt") or {}).get("class_iou", {}).get(name)
    print(f"{name:18s} {VAL_METRICS['class_iou'][name]:.4f}    {last if last is None else f'{last:.4f}'}")
print()
print("mask mAP50        ", VAL_METRICS["mask_map50"])
print("mask mAP50-95     ", VAL_METRICS["mask_map50_95"])
print("head_status       ", PROJECTKV["head_status"])
print("reinitialised     ", PROJECTKV["reinitialised_tensors"])

## 8. Download the deployment bundle

The zip contains `best.pt`, `last.pt`, the IR pair + `metadata.yaml`, `val_metrics.json`,
`run_manifest.json`, `gate_summary.json`, `evidence_report.json`, the frozen dataset
documents, the training log and a `checksums.sha256`. Every file is verified before the
zip is written; `checksums.sha256` lets the deployment host re-verify it independently.

In [ ]:
import os
import shutil
from pathlib import Path

BUNDLE = KV_WORK_DIR / "projectkv_s13_yolov8n_seg_bundle.zip"
assert BUNDLE.is_file(), f"bundle missing: {BUNDLE}"
print(f"{BUNDLE} ({BUNDLE.stat().st_size / (1024 ** 2):.2f} MiB)")

DRIVE_DIR = os.environ.get("KV_DRIVE_DIR", "").strip()
if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path(DRIVE_DIR) / BUNDLE.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BUNDLE, destination)
    print("copied to", destination)
else:
    try:
        from google.colab import files
        files.download(str(BUNDLE))
    except Exception as exc:
        print("automatic download unavailable; copy the bundle manually:", exc)

## 9. Remaining promotion step (AVX2 worker)

The gate's p95 criterion must be measured on the AVX2 inference worker, not on Colab
(spec §2.9/AC18). If `KV_LATENCY_EVIDENCE` was empty, the gate used a provisional Colab CPU
number. On the worker:

```bash
make bench-seg-avx2 SEG_IR=<bundle>/models/yolov8n_seg_openvino_416 \
  SEG_FRAME=<real proof frame> LATENCY_OUT=docs/reports/s13_t1_latency_avx2.json
```

Upload that JSON to this Colab runtime and re-run the driver with
`--skip-training --latency-evidence <that JSON>`: it re-validates `best.pt`, re-evaluates the
gate with the AVX2 p95, re-exports the IR with the passing verdict and rebuilds the bundle,
without spending another 100 epochs. Alternatively call
`evaluate_retraining_gate(metrics=..., metrics_provenance=..., p95_latency_ms=...)` directly.
Do not promote the IR while `gate_summary.json` says `gate_latency_authoritative: false`.

Never commit the IR or `best.pt`: they are gitignored. Promotion records size and SHA-256 in
evidence, per Rule 6.